In [1]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

Crear directorio `csv_cleaned`

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

Cargar archivos csv

In [3]:
df = pd.read_csv(root / "csv_enunciados/ejercicios_enunciados.csv", encoding="utf-8")

In [4]:
def clean_dataframe(df):
    df_sin_metadatos = df[df.aviso == "sin metadatos"]
    df = df[df.aviso.isna()]

    # modify ejercicio column
    df.ejercicio = df.ejercicio.apply(lambda x : x.replace("Opción", "opción")
                                      .replace(".", ",") if pd.notna(x) else x)

    # remove gravity from physics statements
    patron = r"g\s*=\s*9[,.]8\s*m\s*[·.]?\s*s\s*-\s*2\s*"
    mask = df.asignatura == "Física"
    df.loc[mask, "enunciado"] = df.loc[mask, "enunciado"].str.replace(patron, "", regex=True)
    
    # normalize text
    df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto)
                                      if pd.notna(texto) else texto)

    # trim multiple spaces
    df.enunciado = df.enunciado.apply(lambda texto : re.sub(r"[ \t]+", " ", texto)
                                      .strip() if pd.notna(texto) else texto)

    # eliminar , al final del ejercicio
    df.ejercicio = df.ejercicio.str.strip(",")

    # create folder for clean csv
    output_dir = Path(root / "csv_cleaned")
    output_dir.mkdir(exist_ok=True)

    return df

Arreglar ejercicios de Ponencia

In [5]:
def fix_ponencia(row):
    if "PONENCIA" in row["info_ejercicio"]:
        row["convocatoria"] = "Ponencia"
        pos_ejercicio = row["info_ejercicio"].find("EJERCICIO")
        row["ejercicio"] = row["info_ejercicio"][pos_ejercicio:].title()
    return row

In [6]:
df = df.apply(fix_ponencia, axis = 1)

In [7]:
df[df.ejercicio.isna()]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
2556,Mates II,Mates II,2017,NaN,NaN,Matrices Y Determinantes,MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE...,Sea A una matriz 3x3 tal que det(2A)8.\na) ¿C...,source_pdf_files/Mates II/matrices_determinant...,enunciados_ejercicios/Mates II/matrices_determ...,NaN


In [8]:
df.loc[2556, "info_ejercicio"]

'MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE RUC CIC II O Ó 3 .N O PCIÓN B.'

In [9]:
df.loc[2556, ["convocatoria", "ejercicio"]] = ["Reserva 3", "Ejercicio 3, opción B"]

In [10]:
df.sample()

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
1223,Mates CCSS,Mates CCSS,2016,Reserva 2,Ejercicio 4. Opcion B,Contraste De Hipótesis,SOCIALES II. 2016. RESERVA 2. EJERCICIO 4. OPC...,En un artículo de internet se afirma que el nú...,source_pdf_files/Mates CCSS/contraste_hipótesi...,enunciados_ejercicios/Mates CCSS/contraste_hip...,NaN


In [11]:
df = clean_dataframe(df)

Tras la limpieza completa, creo el ID

In [12]:
def create_id(row):
    partes = [row["asignatura"], row["tema"], row["año"], row["convocatoria"]]
    if pd.notna(row["ejercicio"]):
        partes.append(row["ejercicio"])
    id_string = "_".join(str(p) for p in partes).lower()

    # quitar acentos: descomponer y eliminar los caracteres combinantes
    id_string = unicodedata.normalize("NFKD", id_string)
    id_string = "".join(c for c in id_string if not unicodedata.combining(c))

    # dejar solo letras, números y _ (quita comas, puntos, espacios...)
    id_string = re.sub(r"[^a-z0-9]+", "_", id_string).strip("_")
    return id_string

In [13]:
# create id
df["id"] = df.apply(lambda row: create_id(row), axis = 1)
    
# rearrange columns
df = df[["id"] + list(df.columns[:-2])]

Exportar el dataframe limpio

In [14]:
df = (df.sort_values(["asignatura", "tema", "año", "convocatoria", "ejercicio"],
               ascending = [True, True, False, True, True])
               .reset_index(drop = True)
)

In [15]:
df.sample(5)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
4745,quimica_acido_base_2002_junio_ejercicio_4_opci...,Química,Química,2002,Junio,"Ejercicio 4, opción B",Ácido Base,QUÍMICA. 2002. JUNIO. EJERCICIO 4. OPCIÓN B,"Razone, mediante un ejemplo, si al disolver un...",source_pdf_files/Química/ácido_base/2002 - Áci...,enunciados_ejercicios/Química/ácido_base/png/p...
1246,mates_ccss_probabilidad_2023_junio_ejercicio_c5,Mates CCSS,Mates CCSS,2023,Junio,Ejercicio C5,Probabilidad,SOCIALES II. 2023. JUNIO. EJERCICIO C5,Disponemos de una moneda trucada en la que la ...,source_pdf_files/Mates CCSS/probabilidad/2023 ...,enunciados_ejercicios/Mates CCSS/probabilidad/...
2352,mates_ii_integrales_2023_reserva_2_ejercicio_3,Mates II,Mates II,2023,Reserva 2,Ejercicio 3,Integrales,MATEMÁTICAS II. 2023. RESERVA 2. EJERCICIO 3,Sabiendo que F:  definida por: F(x)ex2 es un...,source_pdf_files/Mates II/integrales/2023 - In...,enunciados_ejercicios/Mates II/integrales/png/...
2016,mates_ii_funciones_2000_reserva_3_ejercicio_2_...,Mates II,Mates II,2000,Reserva 3,"Ejercicio 2, opción A",Funciones,MATEMÁTICAS II. 2000. RESERVA 3. EJERCICIO 2. ...,x2\nSea f la función definida x  2 por f(x)...,source_pdf_files/Mates II/funciones/2000 - Fun...,enunciados_ejercicios/Mates II/funciones/png/p...
1154,mates_ccss_matrices_y_determinantes_2014_reser...,Mates CCSS,Mates CCSS,2014,Reserva 3,"Ejercicio 1, opción B",Matrices Y Determinantes,SOCIALES II. 2014 RESERVA 3. EJERCICIO 1. OPCI...,a) Determine los valores de x e y que hacen ci...,source_pdf_files/Mates CCSS/matrices_determina...,enunciados_ejercicios/Mates CCSS/matrices_dete...


In [16]:
df.isna().sum()

id                    0
asignatura            0
asignatura_carpeta    0
año                   0
convocatoria          0
ejercicio             0
tema                  0
info_ejercicio        0
enunciado             4
archivo               0
imagen                0
dtype: int64

In [17]:
df[df.enunciado.isna()]

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
442,fisica_ondas_2016_reserva_3_ejercicio_4_opcion_a,Física,Física,2016,Reserva 3,"Ejercicio 4, opción A",Ondas,FISICA. 2016. RESERVA 3. EJERCICIO 4. OPCIÓN A,NaN,source_pdf_files/Física/movimiento_ondulatorio...,enunciados_ejercicios/Física/movimiento_ondula...
3955,quimica_reacciones_redox_2026_junio_ejercicio_5,Química,Química,2026,Junio,Ejercicio 5,Reacciones Redox,QUÍMICA. 2026. JUNIO. EJERCICIO 5,NaN,source_pdf_files/Química/redox/2026 - Reaccion...,enunciados_ejercicios/Química/redox/png/png_Qu...
4403,quimica_termoquimica_2025_julio_ejercicio_5,Química,Química,2025,Julio,Ejercicio 5,Termoquímica,QUÍMICA.2025. JULIO. EJERCICIO 5,NaN,source_pdf_files/Química/termoquímica/2025 - T...,enunciados_ejercicios/Química/termoquímica/png...
4406,quimica_termoquimica_2025_reserva_2_ejercicio_5,Química,Química,2025,Reserva 2,Ejercicio 5,Termoquímica,QUÍMICA.2025. RESERVA 2. EJERCICIO 5,NaN,source_pdf_files/Química/termoquímica/2025 - T...,enunciados_ejercicios/Química/termoquímica/png...


In [18]:
df.asignatura.unique()

<ArrowStringArray>
['Física', 'Mates CCSS', 'Mates II', 'Química']
Length: 4, dtype: str

In [19]:
df.loc[:, ["asignatura", "tema"]].drop_duplicates()

,asignatura,tema
0,Física,Campo eléctrico y magnético
121,Física,Campo gravitatorio
244,Física,Física cuántica y nuclear
362,Física,Ondas
449,Física,Óptica geométrica
486,Mates CCSS,Contraste De Hipótesis
545,Mates CCSS,Distribución Binomial
546,Mates CCSS,Distribución Binomial
550,Mates CCSS,Funciones CCSS
806,Mates CCSS,Inferencia Estadística


In [20]:
df.tema = df.tema.str.strip()

In [21]:
df.loc[df.tema == "Matrices Y Determinantes", "tema"] = "Matrices y determinantes"

In [22]:
df[df.asignatura == "Mates CCSS"].tema.unique()

<ArrowStringArray>
[         'Contraste De Hipótesis',           'Distribución Binomial',
                  'Funciones CCSS',          'Inferencia Estadística',
        'Matrices y determinantes',                    'Probabilidad',
             'Programación Lineal', 'Sistemas de Ecuaciones Lineales']
Length: 8, dtype: str

In [23]:
df.convocatoria.unique()

<ArrowStringArray>
[             'Julio',              'Junio',          'Reserva 1',
          'Reserva 2',          'Reserva 3',          'Reserva 4',
         'Septiembre', 'Reserva1 Ejercicio',           'Ponencia',
             'Modelo']
Length: 10, dtype: str

In [24]:
df[df.convocatoria == "Reserva1 Ejercicio"]

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
195,fisica_campo_gravitatorio_2020_reserva1_ejerci...,Física,Física,2020,Reserva1 Ejercicio,1,Campo gravitatorio,FISICA. 2020. RESERVA1. EJERCICIO 1,a) Considere dos partículas de igual masa sepa...,source_pdf_files/Física/campo_gravitatorio/202...,enunciados_ejercicios/Física/campo_gravitatori...


In [25]:
df.loc[223, ["convocatoria", "ejercicio"]] = ("Reserva 1", "Ejercicio 1")

In [26]:
df.to_csv(root/"csv_enunciados/enunciados_ejercicios_clean.csv", index = False, encoding="utf-8")

In [27]:
df = pd.read_csv(root / "csv_enunciados/enunciados_ejercicios_clean.csv", encoding="utf-8")

In [28]:
df.sample(3)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
101,fisica_campo_electrico_y_magnetico_2017_reserv...,Física,Física,2017,Reserva 1,"Ejercicio 2, opción A",Campo eléctrico y magnético,FISICA. 2017. RESERVA 1. EJERCICIO 2. OPCIÓN A,"a) Por un hilo recto muy largo, colocado sobre...",source_pdf_files/Física/campo_eléctrico_magnét...,enunciados_ejercicios/Física/campo_eléctrico_m...
2081,mates_ii_geometria_2020_junio_ejercicio_4,Mates II,Mates II,2020,Junio,Ejercicio 4,Geometría,MATEMÁTICAS II. 2020. JUNIO. EJERCICIO 4,"Siendo a0, considera las rectas\nz1 x3 y3 ...",source_pdf_files/Mates II/geometría/2020 - Geo...,enunciados_ejercicios/Mates II/geometría/png/p...
427,fisica_ondas_2018_septiembre_ejercicio_3_opcion_b,Física,Física,2018,Septiembre,"Ejercicio 3, opción B",Ondas,FISICA. 2018. SEPTIEMBRE. EJERCICIO 3. OPCIÓN B,a) ¿Es lo mismo velocidad de vibración que vel...,source_pdf_files/Física/movimiento_ondulatorio...,enunciados_ejercicios/Física/movimiento_ondula...


In [29]:
df.loc[0, :].imagen

'enunciados_ejercicios/Física/campo_eléctrico_magnético/png/png_Física_2025 - Campo eléctrico y magnético/Física_campo_eléctrico_magnético_2025_julio_ejercicio_a.png'